# `lambdaquery` — try it out

`lambdaquery` is a Python package for pulling CMB data out of NASA's [Legacy Archive for Microwave Background Data (LAMBDA)](https://lambda.gsfc.nasa.gov/). The whole public API is three functions:

| | |
|---|---|
| `lq.list_experiments()` | every experiment in the catalog |
| `lq.list_datasets(experiment)` | every dataset under one experiment |
| `lq.fetch_data(experiment, dataset, location)` | download it |

Under the hood it routes downloads to the public `nasa-lambda` S3 mirror where the file exists there, and falls back to the LAMBDA HTTPS server otherwise. That's invisible from the outside — it only affects speed.

## Running this notebook

If you are downloading the code to the library, you can see how it runs using the following steps

```bash
git clone <repo-url> && cd lambdaquery
uv sync
uv run jupyter lab notebooks/lambdaquery_demo.ipynb
```

`uv sync` installs the package in editable mode, so edits to `src/lambdaquery/` are picked up **on the next cell run** — no kernel restart, no reinstall. (The one exception is `data/manifest.yaml`; see section 4.)

Run the cells top to bottom. Nothing here writes into the repo, and the only downloads are two small text files (~50 KB total).

In [1]:
import lambdaquery as lq

## 1. Browse the catalog

The catalog comes from a manifest bundled with the package, so listing is instant and offline — nothing below touches the network until section 3.

In [2]:
experiments = lq.list_experiments()
print(f"{len(experiments)} experiments\n")
for name in experiments:
    print(" ", name)

32 experiments

  ABS
  ACBAR
  ACT
  ARCADE
  Archeops
  BICEP1
  BICEP2
  BOOMERanG
  CAPMAP
  CBI
  CLASS
  COBE/DIRBE
  COBE/DMR
  COBE/FIRAS
  DASI
  Foreground
  IRAS
  MAXIMA
  MSAM
  PIXIE
  POLARBEAR
  Planck
  QMAP
  QMASK
  QUIET
  QUIJOTE
  QUaD
  SPIDER
  SPT
  TRIS
  VSA
  WMAP


In [3]:
# Dataset lists can be long -- COBE/DMR has a couple hundred entries and WMAP has
# nearly 7,000 -- so slice before printing.
datasets = lq.list_datasets("COBE/DMR")
print(f"COBE/DMR: {len(datasets)} datasets, first 20:\n")
for name in datasets[:20]:
    print(" ", name)

print(f"\n(for comparison, WMAP has {len(lq.list_datasets('WMAP'))} entries)")

COBE/DMR: 212 datasets, first 20:

  DMR_CUSTOM_GALAXY_CUT_ECL.TXT
  DMR_CUSTOM_GALAXY_CUT_GAL.TXT
  DMR_DCMB_4YR.FITS
  DMR_DCMB_GALACTIC_4YR.FITS
  DMR_DSMB_4YR.FITS
  DMR_DSMB_GALACTIC_4YR.FITS
  DMR_DUST_4YR.FITS
  DMR_DUST_GALACTIC_4YR.FITS
  DMR_FIRAS_SKYMAP_INFO.FITS
  DMR_FREE_4YR.FITS
  DMR_FREE_GALACTIC_4YR.FITS
  DMR_PIXDIFF_31A_4YR.FITS
  DMR_PIXDIFF_31B_TIME1_4YR.FITS
  DMR_PIXDIFF_31B_TIME2_4YR.FITS
  DMR_PIXDIFF_31B_TIME3_4YR.FITS
  DMR_PIXDIFF_53A_4YR.FITS
  DMR_PIXDIFF_53B_4YR.FITS
  DMR_PIXDIFF_90A_4YR.FITS
  DMR_PIXDIFF_90B_4YR.FITS
  DMR_SKYMAP_31A_4YR.FITS

(for comparison, WMAP has 6860 entries)


## 2. Download something

`lq.fetch_data(experiment, dataset, location)` returns a `Path` for a file leaf, or a `list[Path]` for a group.

Files land under `location` mirroring their LAMBDA `/data/...` path, which means a file referenced by several groups is stored — and downloaded — exactly once. Downloads stream to a `.part` temp file and are renamed into place at the end, so an interrupted download never looks complete.

In [4]:
file_loc = lq.fetch_data('QMAP', '1Ka12_cmbmap.dat', './')
file_loc

PosixPath('data/suborbital/QMAP/1Ka12_cmbmap.dat')